In [1]:
import numpy as np
from pathlib import Path
from generator import systemInit, setup_ham_rse, calculate_spectral_density
from tqdm.auto import tqdm
import h5py

import ipywidgets as widgets
from ipywidgets import interact, Dropdown, fixed, IntSlider

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
sns.set_theme(style="white", context="notebook")

# 2. Fine-tune specific font sizes and behaviors
plt.rcParams.update({
    "font.family": "serif",        # Use serif for a formal/academic look
    "font.size": 9,               # Base font size
    "axes.titlesize": 18,          # Subplot titles (your .set(title=...) calls)
    "axes.labelsize": 14,          # X and Y axis labels
    "xtick.labelsize": 10,         # Size of numbers on X-axis
    "ytick.labelsize": 10,         # Size of numbers on Y-axis
    "legend.fontsize": 14,         # Legend text
    "figure.titlesize": 20,        # Overall figure title
    "axes.labelpad": 10,           # Distance between label and axis
    "mathtext.fontset": "cm",      # Computer Modern (LaTeX look for math)
    "savefig.dpi": 300             # High-res exports
})

REAL_PART_CMP = sns.color_palette("RdBu", as_cmap=True)

In [3]:
OUTDIR = Path("conv_data")
if not OUTDIR.exists():
    print("Err: dir '{}' does NOT exist".format(OUTDIR))
    CWD = Path.cwd()
    print('Create \'{}\' at \'{}\' '.format(OUTDIR, CWD))
    # OUT_DIR.mkdir(parents=False, exist_ok=False)
    # print('####\'{}\' created'.format(OUT_DIR))
else:
    print("dir '{}' exists!".format(OUTDIR))

dir 'conv_data' exists!


In [4]:
def load_data():
    return h5py.File(OUTDIR / "RSE_data.h5", 'r')

In [5]:
def render_plot(outputs, N, eta, E, part, site, nn, nn_pair, g_minmax=False):
    energies = outputs.attrs['E']
    current_N = outputs[f"N_{N}"]
    RSE = current_N[f"eta_{eta:.1e}"]
    if g_minmax: # if global min/max
        vmin = outputs.attrs['vmin']
        vmax = outputs.attrs['vmax']
    elif not g_minmax:
        vmin = np.min([np.real(RSE), np.imag(RSE)])
        vmax = np.max([np.real(RSE), np.imag(RSE)])
    else:
        raise ValueError("'g_minmax' must be boolean")
    vmax = np.abs([vmin, vmax]).max()
    vmin = -vmax
        
    xyz = np.asanyarray(current_N['xyz'])
    x,y,z = xyz.T
    
    E_idx = np.argwhere(energies == E)[0,0]
    part = part.lower()
    if (part=="real"):
        RSE_part = np.real(RSE)
    elif (part=="imag"):
        RSE_part = np.imag(RSE)
        
    site_coupling = RSE_part[E_idx, site, :]
    # print(f"{RSE_part.shape = }")
    
    site_xyz = xyz[[site], :]
    elec_xyz = xyz[:N, :]
    diff = xyz[:, None, :] - site_xyz[None, :, :]
    diff_site = site_xyz[:, None, :] - elec_xyz[None, :, :]
    dists = np.linalg.norm(diff, axis=2).squeeze()
    dists_site = np.linalg.norm(diff_site, axis=2).squeeze()
    
    ylim = (vmin, vmax)
    xlim = (-1, np.linalg.norm(xyz[0, :] - xyz, axis=-1).max()
        )
    
    fig, axes = plt.subplots(1,2, figsize=(10,4), dpi=150)
    
    axes[0].scatter(dists[N:], site_coupling[N:], marker=".", color="r", s=15)
    axes[0].scatter(dists[:N], site_coupling[:N], marker="+", color="b", s=30)
    axes[0].axhline(0, color="k", linestyle="--", linewidth=.8)
    
    axes[0].set(xlabel=r"$r$ $[\mathrm{\AA}]$",
                ylabel=rf"$\Sigma_{{{site}, j}}$",
                xlim=xlim, ylim=ylim,
                title="Coupling Vs. distance",
        )
    
    sc = axes[1].scatter(x,y,c=site_coupling, vmin=vmin, vmax=vmax, cmap="RdBu", s=100/N)
    
    axes[1].set(xlabel="$x$",
                ylabel="$y$",
                xticklabels="",
                yticklabels="",
                title=rf"Coupling in 2D, $\Sigma_{{{site}, j}}$"
        
        )
    
    farthest_dist = 0
    for s in np.arange(site-nn, site+nn+1, 1):
        if (0 <= s) and (s < N):
            if s != site:
                farthest_dist = np.max([farthest_dist, dists[s]])
                axes[1].text(x=x[s], y=y[s]-1, s=s, horizontalalignment="right", verticalalignment="top")
            elif (s==site):
                axes[1].annotate(site, xy=(x[site], y[site]), xycoords="data", 
                                 xytext=(x[site]+2*N/9, y[site]+3*N/9), 
                                 arrowprops=dict(color='k', width=0.10, headwidth=3.0, headlength=3.0)
                    )
    farthest_nn_mask = np.nonzero(np.isclose(dists_site, farthest_dist, atol=1e-2))[0]
    if not nn_pair:
        farthest_nn_mask = farthest_nn_mask[[0]]
    axes[0].axvline(farthest_dist, color='k', linestyle='--', lw=1, alpha=0.5)
    coupling_for_farthest_nn = site_coupling[farthest_nn_mask]
    for coup in coupling_for_farthest_nn:
        axes[0].annotate(text=f" {coup:.1f}", xy=(farthest_dist, coup), xytext=(farthest_dist, coup))
    
    axes[1].axis('off')
    axes[0].grid(True, which="both", linestyle='-', lw=0.2)
    
    divider = make_axes_locatable(axes[1])
    cax = divider.append_axes('right', size='5%', pad=0.1)
    fig.colorbar(sc, cax)
    
    fig.canvas.draw()

In [6]:
data = load_data()
N_options = data.attrs['N']
eta_options = data.attrs['ETA'].round(7)
energy_options = data.attrs['E']
E0_idx = data.attrs['E0_idx']

N_menu = widgets.Dropdown(value=N_options.min(), options=N_options, description='Tiling (N)')
part_menu = widgets.RadioButtons(index=0, options=dict(Imaginary='imag', Real='real'))
E_slider = widgets.FloatSlider(value=0, min=energy_options.min(), max=energy_options.max(), description="Energy [eV]", readout_format=".1f")
eta_menu = widgets.Dropdown(value=eta_options.max(), options=eta_options, description='Eta')
site_slider = widgets.IntSlider(min=0, max=int(N_options.min()-1), step=1, description='Site:')
neighbour_slider = widgets.IntSlider(min=0, max=np.abs([site_slider.max - site_slider.value, site_slider.value]).max(),step=1, description="Neighbours")
show_nn_pair_box = widgets.Checkbox(value=False, description="Show pairs of NN")
global_min_max = widgets.Checkbox(value=False, description="Use global min/max")

def update_site_range(*args):
    new_n = int(N_menu.value)
    site_slider.max = new_n -1
    if site_slider.value > site_slider.max:
        site_slider.value = site_slider.max
def update_nn_range(*args):
    new_site = int(site_slider.value)
    neighbour_slider.max = np.abs([site_slider.max - new_site, new_site]).max()
    if site_slider.value > site_slider.max:
        site_slider.value = site_slider.max

N_menu.observe(update_site_range, 'value')
site_slider.observe(update_nn_range, 'value')


In [7]:
UI = widgets.VBox([
    widgets.HBox([N_menu, eta_menu]),
    widgets.HBox([E_slider, site_slider, neighbour_slider]),
    widgets.HBox([global_min_max, show_nn_pair_box, part_menu]),
])
# N, eta, E, part, site, nn, nn_pair, g_minmax
out = widgets.interactive_output(render_plot, dict(
    outputs=widgets.fixed(data),
    N=N_menu,
    eta=eta_menu,
    E=E_slider,
    part=part_menu,
    site=site_slider,
    nn=neighbour_slider,
    nn_pair=show_nn_pair_box,
    g_minmax=global_min_max,
))

display(UI, out)

Output()